Датасет был загружен и были показаны первые 5 строк

In [13]:
import pandas as pd 
df=pd.read_csv('../data/ml-1m/ratings.dat',sep='::',names=['user_id','movie_id','rating','timestamp'],engine='python')
df.head()

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


Далее была проверена размерность датасета

In [14]:
print(f'Количество строк в датасете {df.shape[0]}, количество признаков - {df.shape[1]}')

Количество строк в датасете 1000209, количество признаков - 4


Далее выводится информация по каждой колонке

In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000209 entries, 0 to 1000208
Data columns (total 4 columns):
 #   Column     Non-Null Count    Dtype
---  ------     --------------    -----
 0   user_id    1000209 non-null  int64
 1   movie_id   1000209 non-null  int64
 2   rating     1000209 non-null  int64
 3   timestamp  1000209 non-null  int64
dtypes: int64(4)
memory usage: 30.5 MB


Пропущенные значения в датасете отсутствуют. Все столбцы представлены в числовом формате. Датасет содержит информацию об оценках фильмов пользователями:
- user_id — уникальный идентификатор пользователя
- movie_id — уникальный идентификатор фильма
- rating — оценка, выставленная пользователем фильму
- timestamp — время выставления оценки в формате Unix timestamp

Далее провяется количество пропусков методом для точной проверки


In [16]:
df.isna().sum()

user_id      0
movie_id     0
rating       0
timestamp    0
dtype: int64

Подтвердилось - пропуски отсутствуют, далее будут проверены дубликаты

In [17]:
df.duplicated().sum()

np.int64(0)

Дубликаты аналогично отсутствуют. Теперь необходимо проверить состав класса рейтинга

In [19]:
df.rating.value_counts().sort_index()

rating
1     56174
2    107557
3    261197
4    348971
5    226310
Name: count, dtype: int64

Сейчас класс рейтинга находится в явном дисбалансе, но в таком виде он и не будет использоваться далее. Была введена новая колонка `interaction`
Правило:
Если оценка >=4, взаимодействие положительное (1)
Остальные ситуации для матрицы нас не интересуют


In [22]:
df = df[df["rating"] >= 4].copy()
df["interaction"] = 1

In [24]:
df.head(5)

,user_id,movie_id,rating,timestamp,interaction
0,1,1193,5,978300760,1
3,1,3408,4,978300275,1
4,1,2355,5,978824291,1
6,1,1287,5,978302039,1
7,1,2804,5,978300719,1


In [25]:
print("Пользователей:", df["user_id"].nunique())
print("Фильмов:", df["movie_id"].nunique())
print("Положительных взаимодействий:", len(df))

Пользователей: 6038
Фильмов: 3533
Положительных взаимодействий: 575281


Далее необходимо привести timestamp к нормальной дате

In [26]:
df["datetime"] = pd.to_datetime(df["timestamp"], unit="s")

df[["timestamp", "datetime"]].head()

,timestamp,datetime
0,978300760,2000-12-31 22:12:40
3,978300275,2000-12-31 22:04:35
4,978824291,2001-01-06 23:38:11
6,978302039,2000-12-31 22:33:59
7,978300719,2000-12-31 22:11:59


Далее были просмотрены диапазоны дат

In [27]:
print("Первая оценка:", df["datetime"].min())
print("Последняя оценка:", df["datetime"].max())

Первая оценка: 2000-04-25 23:05:32
Последняя оценка: 2003-02-28 17:49:50


In [35]:
user_interactions = df.groupby("user_id").size()

user_interactions.describe()

count    6038.000000
mean       95.276747
std       105.005005
min         1.000000
25%        27.000000
50%        58.000000
75%       124.000000
max      1435.000000
dtype: float64

In [36]:
print("Пользователей с одним положительным взаимодействием:",
      (user_interactions == 1).sum())

Пользователей с одним положительным взаимодействием: 1


In [37]:
user_interactions = df.groupby("user_id").size()

valid_users = user_interactions[user_interactions >= 2].index

df = df[df["user_id"].isin(valid_users)].copy()

In [38]:
df.groupby("user_id").size().min()

np.int64(2)

Данные были разделены на обучающую и валидационную выборки по времени. Для каждого пользователя взаимодействия с максимальным timestamp были отнесены к validation, а все более ранние взаимодействия — к train. Пользователи, имеющие взаимодействия только в один момент времени, были исключены, так как для них невозможно сформировать временное разделение без утечки информации. Дополнительно была выполнена проверка, что для каждого пользователя максимальное время взаимодействия в train строго меньше минимального времени взаимодействия в validation

In [49]:
user_timestamps = df.groupby("user_id")["timestamp"].nunique()

valid_users = user_timestamps[user_timestamps >= 2].index
df = df[df["user_id"].isin(valid_users)].copy()

max_timestamp = df.groupby("user_id")["timestamp"].transform("max")

val_df = df[df["timestamp"] == max_timestamp].copy()
train_df = df[df["timestamp"] < max_timestamp].copy()

train_last = train_df.groupby("user_id")["timestamp"].max()
val_first = val_df.groupby("user_id")["timestamp"].min()

print("Корректность временного разделения:",
      (train_last < val_first).all())

print("Train:", train_df.shape)
print("Validation:", val_df.shape)

print("Пользователей в train:", train_df["user_id"].nunique())
print("Пользователей в validation:", val_df["user_id"].nunique())

Корректность временного разделения: True
Train: (565861, 6)
Validation: (9417, 6)
Пользователей в train: 6036
Пользователей в validation: 6036


Считаем популярность фильмов только по train

In [55]:
popularity = (
    train_df
    .groupby("movie_id")
    .size()
    .sort_values(ascending=False)
)
popularity.head(10)

movie_id
2858    2817
260     2582
1196    2486
1198    2242
2028    2241
593     2226
2571    2140
2762    2123
1210    2111
527     2050
dtype: int64

In [58]:
popular_items = popularity.index.tolist()

Посмотрим историю одного пользователя

In [59]:
user_id = 1

seen_items = set(
    train_df.loc[
        train_df["user_id"] == user_id,
        "movie_id"
    ]
)

seen_items

{1,
 150,
 260,
 527,
 531,
 588,
 594,
 595,
 608,
 783,
 919,
 938,
 1022,
 1028,
 1029,
 1035,
 1097,
 1193,
 1207,
 1246,
 1270,
 1287,
 1545,
 1566,
 1721,
 1836,
 1907,
 1961,
 1962,
 2018,
 2028,
 2294,
 2355,
 2398,
 2692,
 2762,
 2791,
 2797,
 2804,
 2918,
 3105,
 3114,
 3186,
 3408}

Рекомендуем популярные фильмы, которых пользователь ещё не видел

In [60]:
recommendations = [
    movie_id
    for movie_id in popular_items
    if movie_id not in seen_items
][:10]

recommendations

[2858, 1196, 1198, 593, 2571, 1210, 318, 589, 858, 110]